# Net2Brain


## Installation (if not already installed)

In [1]:
# The 'CCN23' branch no longer exists in the repo (verified 2026-09-09) -> install from main instead.
# Note: main branch's API has diverged in places from the CCN23 tutorial (see fixes below,
# especially in the VPA section).
# !pip install -U git+https://github.com/cvai-roig-lab/Net2Brain
# !pip install nilearn==0.9.2


## 0. Basic setup (local path + single device)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, glob, shutil
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# the path can point either to an extracted folder or to a ZIP archive.
# read from an env var first so the notebook doesn't ship with someone else's local path.
data_dir = Path(os.environ.get('N2B_DATA_DIR', r'C:\Users\662\Downloads\subj01-20260908T230632Z-1-001.zip'))  # <-- CHANGE if needed

def check_free_space(path, min_gb=15):
    """fail fast with a clear message instead of dying mid-extraction with
    'OSError: No space left on device' (this is what killed the run last time,
    in the feature-extraction cell below)."""
    usage = shutil.disk_usage(Path(path).anchor or '/')
    free_gb = usage.free / (1024 ** 3)
    if free_gb < min_gb:
        raise OSError(
            f"Only {free_gb:.1f} GB free on the drive containing {path}. "
            f"Feature extraction over {min_gb}+ layers x 1000 images needs a good deal of "
            f"scratch space (uncompressed activations before consolidation). "
            f"Free up space or point data_dir/save_path at a drive with more room."
        )
    return free_gb

print(f'Free space near data_dir: {check_free_space(data_dir):.1f} GB')

if data_dir.is_file() and data_dir.suffix.lower() == '.zip':
    extracted_dir = data_dir.with_suffix('')
    subj_dir = extracted_dir / 'subj01'
    if not subj_dir.is_dir():
        print(f'Extracting archive to: {extracted_dir}')
        with zipfile.ZipFile(data_dir) as archive:
            archive.extractall(extracted_dir)
    data_dir = extracted_dir
elif not data_dir.is_dir():
    raise FileNotFoundError(f'Folder or ZIP archive not found: {data_dir}')

# use one device variable throughout the notebook.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
print('data_dir:', data_dir)

subj = '01'
sml_stim = data_dir / f'subj{subj}' / 'sml_images'
sml_fmri = data_dir / f'subj{subj}' / 'sml_fmri'

if not sml_stim.is_dir():
    raise FileNotFoundError(f'Image folder not found: {sml_stim}')

sml_img_list = sorted(os.listdir(sml_stim))
print(f'Total images: {len(sml_img_list)}')


OSError: Only 7.2 GB free on the drive containing C:\Users\662\Downloads\subj01-20260908T230632Z-1-001.zip. Feature extraction over 15+ layers x 1000 images needs a good deal of scratch space (uncompressed activations before consolidation). Free up space or point data_dir/save_path at a drive with more room.

## 1-2. Load Taskonomy NN + run the Sem model (V1, V2, V3, PPA)

Previously only `layer4` was extracted. We now use all encoder layers so that the layer-wise plots in sections 3-5 can be created.

In [ ]:
from net2brain.feature_extraction import FeatureExtractor
import net2brain.evaluations.encoding as encoding

model_name = 'class_object'  # Semantic model (Sem NN)

fx_model = FeatureExtractor(model=model_name, netset='Taskonomy', device=device)

layers_to_extract = 'top_level'

check_free_space(data_dir, min_gb=5)

ft_path = f'sml_feats_{model_name}'
# NOTE: extract() has no 'save_format' argument (it always writes .npz) -- passing
# one used to be silently swallowed by **kwargs and do nothing.
fx_model.extract(
    data_path=sml_stim,
    save_path=ft_path,
    layers_to_extract=layers_to_extract,
)

# resolve the actual layer names now used for downstream cells, from the saved files.
layers_to_extract = sorted(
    f.stem.replace('consolidated_', '')
    for f in Path(ft_path).glob('consolidated_*.npz')
)
print('Extracted layers:', layers_to_extract)


In [ ]:
roi_path = str(sml_fmri)

# n_components must not exceed the number of samples in the training fold
n_train = int(len(sml_img_list) * 0.8)
n_components = min(70, max(2, n_train - 1))
print('n_components used:', n_components)

model_brain_df, model_brain_corr = encoding.linear_encoding(
    ft_path, roi_path, model_name,
    trn_tst_split=0.8,
    n_folds=3,
    n_components=n_components,
    batch_size=300,
    return_correlations=True,
)

rois_of_interest = ['rh_V1_fmri', 'rh_V2_fmri', 'rh_V3_fmri', 'rh_PPA_fmri']
focus_df = model_brain_df[model_brain_df['ROI'].isin(rois_of_interest)]
focus_df[['ROI', 'Layer', 'R']]


### fsaverage visualization (nilearn) - as in the original tutorial

No changes from the original, except that `model_brain_dict` is now built from the fresh `model_brain_df` (after multi-layer extraction there will be several rows per ROI; for an fsaverage map, a specific layer is usually selected, such as the last one).

In [ ]:
from nilearn import datasets, plotting
import matplotlib.colors as plc

roi_idx = np.load((Path(data_dir) / f'subj{subj}' / 'sml_roi_idx_map.npy'), allow_pickle=True)[()]

fsaverage = datasets.fetch_surf_fsaverage('fsaverage')
masks_dir = Path(data_dir) / f'subj{subj}' / 'roi_masks'
rh_fsaverage = np.load((masks_dir / 'rh.all-vertices_fsaverage_space.npy'), allow_pickle=True)
fs_idx = np.where(rh_fsaverage)[0]

# use the last extracted layer for the brain map (change it to any other layer if needed).
last_layer = layers_to_extract[-1]
layer_df = model_brain_df[model_brain_df['Layer'] == last_layer]
model_brain_dict = dict(zip(layer_df.ROI, layer_df.R))

plot_data = np.zeros(rh_fsaverage.shape)
for roi_key, dict_key in [('V1','rh_V1_fmri'), ('V2','rh_V2_fmri'),
                          ('V3','rh_V3_fmri'), ('PPA','rh_PPA_fmri')]:
    plot_data[fs_idx[roi_idx[roi_key]]] = np.ones(fs_idx[roi_idx[roi_key]].shape) * model_brain_dict[dict_key]

view = plotting.view_surf(
    surf_mesh=fsaverage['infl_right'],
    surf_map=plot_data, bg_map=fsaverage['sulc_right'],
    threshold=1e-14, colorbar=True, symmetric_cmap=False,
    cmap=plt.get_cmap('twilight_shifted')
)
view


## 3. Layer-wise plot WITHOUT alphabetical sorting

Preserve the actual extraction order (`layers_to_extract`) with `pd.Categorical`, instead of letting matplotlib/pandas sort strings alphabetically.

In [ ]:
def ordered_layer_plot(df, roi, layer_order, ax=None, label=None, with_std=None):
    """df: dataframe with ROI, Layer, R columns (and optional <with_std>)."""
    sub = df[df['ROI'] == roi].copy()
    sub['Layer'] = pd.Categorical(sub['Layer'], categories=layer_order, ordered=True)
    sub = sub.sort_values('Layer')

    ax = ax or plt.gca()
    x = range(len(sub))
    ax.plot(x, sub['R'], marker='o', label=label or roi)
    if with_std is not None:
        ax.fill_between(x, sub['R'] - sub[with_std], sub['R'] + sub[with_std], alpha=0.2)
    ax.set_xticks(list(x))
    ax.set_xticklabels(sub['Layer'], rotation=45, ha='right')
    return ax

fig, ax = plt.subplots(figsize=(8, 5))
ordered_layer_plot(model_brain_df, 'rh_V1_fmri', layers_to_extract, ax=ax)
ax.set_ylabel('Pearson R')
ax.set_title(f'{model_name}: R by layer (V1), extraction order')
plt.tight_layout()
plt.show()


## 4. Add standard deviation to the plot

CHECK: the structure of `model_brain_corr` (the second object returned by `linear_encoding`) may differ between package versions. It is printed first so you can compare it and adjust `key_variants` in `attach_std` if needed.

In [ ]:
print(type(model_brain_corr))
if isinstance(model_brain_corr, dict):
    print(list(model_brain_corr.keys())[:5])


In [ ]:
def attach_std(df, corr_obj):
    stds = []
    for _, row in df.iterrows():
        key_variants = [(row['Layer'], row['ROI']), row['ROI']]
        vals = None
        for k in key_variants:
            if isinstance(corr_obj, dict) and k in corr_obj:
                vals = corr_obj[k]
                break
        stds.append(np.std(vals) if vals is not None else np.nan)
    df = df.copy()
    df['R_std'] = stds
    return df

model_brain_df_std = attach_std(model_brain_df, model_brain_corr)

fig, ax = plt.subplots(figsize=(8, 5))
ordered_layer_plot(model_brain_df_std, 'rh_V1_fmri', layers_to_extract,
                    ax=ax, with_std='R_std')
ax.set_ylabel('Pearson R')
ax.set_title(f'{model_name}: R +/- std by layer (V1)')
plt.tight_layout()
plt.show()


## 5. Layer-wise similarity: V1+V3 and V2+V3, together with PPA

The dataset has no `V4` ROI (only V1/V2/V3/PPA), which is probably a typo in the specification. Below, we use V1+V3 and V2+V3, each together with PPA. Change the ROI names if V4 is available in your data.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

pairs = [('rh_V1_fmri', 'rh_V3_fmri'), ('rh_V2_fmri', 'rh_V3_fmri')]
for ax, pair in zip(axes, pairs):
    for roi in list(pair) + ['rh_PPA_fmri']:
        ordered_layer_plot(model_brain_df_std, roi, layers_to_extract,
                            ax=ax, label=roi, with_std='R_std')
    ax.legend()
    ax.set_ylabel('Pearson R')
    ax.set_title(' + '.join(pair) + ' + PPA')

plt.tight_layout()
plt.show()


## 6. Variance Partitioning Analysis (VPA)

VPA works with RDMs rather than raw features. First, we build model RDMs (2D/3D/Sem) and brain RDMs for each ROI, then run `VPA`.

CHECK: this section was not in the original notebook. The code was assembled from the documentation (https://net2brain.readthedocs.io/en/latest/evaluation.html#variance-partitioning-analysis-vpa). Check the column names/keys using the `print(...)` statements.

In [ ]:
from net2brain.rdm_creation import RDMCreator

def make_model_rdms(feat_path, save_path):
    creator = RDMCreator(verbose=True, device=device)
    return creator.create_rdms(feature_path=feat_path, save_path=save_path,
                                save_format='npz')

def extract_features(model_name, layers='top_level'):
    fx = FeatureExtractor(model=model_name, netset='Taskonomy', device=device)
    check_free_space(data_dir, min_gb=5)
    ft_path = f'sml_feats_{model_name}'
    fx.extract(data_path=sml_stim, save_path=ft_path, layers_to_extract=layers)
    return ft_path

ft_2d  = extract_features('segment_unsup2d')
ft_3d  = extract_features('reshading')
ft_sem = ft_path  # already extracted in sections 1-2 (class_object)

rdm_2d_path  = make_model_rdms(ft_2d,  'rdms_2d')
rdm_3d_path  = make_model_rdms(ft_3d,  'rdms_3d')
rdm_sem_path = make_model_rdms(ft_sem, 'rdms_sem')


In [ ]:
# brain RDM from fMRI patterns (1 - pearson correlation between images)
def fmri_to_rdm(roi_npy_path):
    resp = np.load(roi_npy_path)   # expected shape: [n_images, n_voxels]
    corr = np.corrcoef(resp)
    return 1 - corr

roi_rdm_dir = Path('roi_rdms')
roi_rdm_dir.mkdir(exist_ok=True)

roi_rdms = {}
roi_rdm_paths = {}
for roi_file in glob.glob(str(sml_fmri / '*.npy')):
    roi_name = Path(roi_file).stem
    rdm = fmri_to_rdm(roi_file)
    roi_rdms[roi_name] = rdm

    out_path = roi_rdm_dir / f'{roi_name}.npz'
    np.savez(out_path, rdm=rdm)
    roi_rdm_paths[roi_name] = str(out_path)

print(list(roi_rdms.keys()))


In [ ]:
from net2brain.evaluations.variance_partitioning_analysis import VPA
from net2brain.evaluations.plotting import Plotting

def pick_layer_rdm(rdm_dir, layer_name=None):
    """VPA needs a path to ONE layer's RDM file, not the whole per-layer folder.
    Defaults to the last extracted layer, matching the convention used in section 7."""
    rdm_dir = Path(rdm_dir)
    files = sorted(rdm_dir.glob('RDM_*.npz'))
    if not files:
        raise FileNotFoundError(f'No RDM files found in {rdm_dir}')
    if layer_name is not None:
        match = [f for f in files if f.stem == f'RDM_{layer_name.replace(".", "_")}']
        if match:
            return str(match[0])
    return str(files[-1])

independent_variables = [
    pick_layer_rdm(rdm_2d_path),
    pick_layer_rdm(rdm_3d_path),
    pick_layer_rdm(rdm_sem_path),
]
variable_names = ['2D', '3D', 'Semantic']

dependent_variable = [roi_rdm_paths['rh_V1_fmri']]  # example for one ROI (note: list!)

VPA_eval = VPA(dependent_variable, independent_variables, variable_names)
vpa_df = VPA_eval.evaluate(average_models=True)
print(vpa_df.columns)  # CHECK the actual column names
vpa_df


In [ ]:
plotter = Plotting(vpa_df)
plotter.plotting_components()   # fMRI (no time dimension): bar chart + SEM + significance
plt.show()


## 7. Combine models of the same "type" into one RDM (2D / 3D / Sem)

Average the RDM of the same encoder layer across all models in each group. CHECK: replace the placeholder output-layer name `'layer4'` with the actual name printed by `fx_model.get_all_layers()`.

In [ ]:
# verify the actual final-layer name for EACH model before averaging
def get_final_layer_name(model_name):
    fx = FeatureExtractor(model=model_name, netset='Taskonomy', device=device)
    layers = [layer for layer in fx.get_all_layers() if layer]
    print(f'{model_name}: available layers -> {layers}')
    return layers[-1]  # take the actual last encoder layer, not a hardcoded guess

groups = {
    '2D':  ['segment_unsup2d', 'inpainting', 'keypoints2d', 'jigsaw', 'autoencoding', 'denoising'],
    '3D':  ['reshading', 'curvature', 'depth_euclidean', 'keypoints3d', 'normal'],
    'Sem': ['class_object', 'class_scene', 'segment_semantic'],
}

# resolve the correct final layer per model instead of assuming 'layer4' for all
final_layers = {m: get_final_layer_name(m) for group in groups.values() for m in group}
print('\nResolved final layers per model:', final_layers)


def group_rdm(model_names, final_layers):
    mats = []
    shapes = []

    for m in model_names:
        encoder_layer = final_layers[m]
        ft = f'sml_feats_{m}'

        # reuse features already extracted in sections 1-2 for class_object (Sem model)
        if not Path(ft).exists():
            fx = FeatureExtractor(model=m, netset='Taskonomy', device=device)
            check_free_space(data_dir, min_gb=2)  # only 1 layer requested, so this is cheap
            # extract() has no 'save_format' arg -- removed (it was silently ignored)
            fx.extract(data_path=sml_stim, save_path=ft, layers_to_extract=[encoder_layer])
        else:
            print(f'{m}: reusing existing features at {ft}')

        creator = RDMCreator(verbose=False, device=device)
        rdm_dir = creator.create_rdms(
            feature_path=ft,
            save_path=f'rdm_{m}',
            save_format='npz'
        )

        # look for the actual .npz file
        rdm_dir = Path(rdm_dir)
        npz_files = sorted(rdm_dir.rglob('*.npz'))
        if not npz_files:
            files = list(rdm_dir.rglob('*')) if rdm_dir.exists() else []
            raise FileNotFoundError(
                f'No .npz files found in RDM folder: {rdm_dir.resolve()}\n'
                f'Found: {files}'
            )

        expected_stem = f"RDM_{encoder_layer.replace('.', '_')}"
        matching_files = [p for p in npz_files if p.stem == expected_stem]
        npz_path = matching_files[0] if matching_files else npz_files[0]
        print(m, 'RDM file:', npz_path.resolve())

        loaded = np.load(npz_path)
        print(m, 'npz keys:', loaded.files)
        key = 'rdm' if 'rdm' in loaded.files else loaded.files[0]
        rdm_matrix = loaded[key]

        mats.append(rdm_matrix)
        shapes.append(rdm_matrix.shape)

    # sanity check: all RDMs in the group must have the same shape before averaging
    if len(set(shapes)) > 1:
        raise ValueError(
            f'RDM shape mismatch within group {model_names}: {dict(zip(model_names, shapes))}\n'
            'Cannot average RDMs of different sizes — check the number of extracted images per model.'
        )

    return np.mean(np.stack(mats, axis=0), axis=0)


rdm_2d_avg  = group_rdm(groups['2D'],  final_layers)
rdm_3d_avg  = group_rdm(groups['3D'],  final_layers)
rdm_sem_avg = group_rdm(groups['Sem'], final_layers)

np.savez('rdm_group_2D.npz',  rdm=rdm_2d_avg)
np.savez('rdm_group_3D.npz',  rdm=rdm_3d_avg)
np.savez('rdm_group_Sem.npz', rdm=rdm_sem_avg)


## 8. UVP (Unique Variance Partitioning) for all ROIs

Run VPA with three averaged group RDMs against each ROI and collect only the unique variance of each group (`y1`, `y2`, `y3`).

In [ ]:
independent_variables = ['rdm_group_2D.npz', 'rdm_group_3D.npz', 'rdm_group_Sem.npz']
variable_names = ['2D', '3D', 'Sem']

uvp_rows = []
for roi_name, roi_rdm in roi_rdms.items():
    vpa_eval = VPA([roi_rdm_paths[roi_name]], independent_variables, variable_names)
    df = vpa_eval.evaluate(average_models=True)

    # confirm the 'Variable' column exists and contains the expected labels
    if 'Variable' not in df.columns:
        raise KeyError(
            f"[{roi_name}] Expected column 'Variable' not found. "
            f"Actual columns: {list(df.columns)}"
        )

    df = df.query("Variable in ['y1', 'y2', 'y3']").copy()
    if df.empty:
        raise ValueError(
            f"[{roi_name}] No rows matched y1/y2/y3. "
            f"Actual Variable values: {vpa_eval.evaluate(average_models=True)['Variable'].unique()}"
        )

    df['ROI'] = roi_name
    uvp_rows.append(df)

uvp_df = pd.concat(uvp_rows, ignore_index=True)

# automatically find the column with variance values
excluded_columns = {'Variable', 'ROI'}
numeric_columns = [
    column for column in uvp_df.select_dtypes(include=np.number).columns
    if column not in excluded_columns
]
preferred_columns = [
    column for column in numeric_columns
    if any(word in column.lower() for word in ['value', 'variance', 'unique', 'explained', 'score', 'r2'])
]

if len(preferred_columns) == 1:
    value_col = preferred_columns[0]
elif len(numeric_columns) == 1:
    value_col = numeric_columns[0]
elif preferred_columns:
    value_col = preferred_columns[0]
else:
    raise ValueError(
        'Could not uniquely find the column with variance values. '
        f'Numeric columns: {numeric_columns}'
    )

# also look for a std/SEM/error column, so the plot can show error bars
std_candidates = [
    column for column in numeric_columns
    if any(word in column.lower() for word in ['std', 'sem', 'error', 'ci'])
]
std_col = std_candidates[0] if std_candidates else None

print('All columns:', list(uvp_df.columns))
print('Variance value column:', value_col)
print('Error/std column:', std_col if std_col else 'NOT FOUND — will need bootstrap for error bars')
uvp_df


In [ ]:
print('Using value column:', value_col)
print('Using error column:', std_col if std_col else 'none available')

# build the main pivot table (values)
pivot = uvp_df.pivot(index='ROI', columns='Variable', values=value_col)
pivot = pivot.rename(columns={'y1': '2D', 'y2': '3D', 'y3': 'Sem'})

# enforce a fixed column order so colors/order stay consistent even if a column is missing
expected_cols = ['2D', '3D', 'Sem']
pivot = pivot.reindex(columns=[c for c in expected_cols if c in pivot.columns])

# build the matching error table (if std_col was found or bootstrapped earlier)
if std_col:
    pivot_err = uvp_df.pivot(index='ROI', columns='Variable', values=std_col)
    pivot_err = pivot_err.rename(columns={'y1': '2D', 'y2': '3D', 'y3': 'Sem'})
    pivot_err = pivot_err.reindex(columns=pivot.columns)  # match value columns exactly
else:
    pivot_err = None

fig, ax = plt.subplots(figsize=(9, 5))
pivot.plot(
    kind='bar',
    yerr=pivot_err,
    capsize=4,
    
    ax=ax,
    color=['#1f77b4', '#ff7f0e', '#2ca02c'][:len(pivot.columns)]
)
ax.set_ylabel('Unique variance explained')
ax.set_title('UVP: 2D vs 3D vs Semantic by ROI\n(available ROIs only: V1, V2, V3, PPA)')
ax.set_xlabel('')
plt.xticks(rotation=0)
plt.legend(title='Model group')
plt.tight_layout()
plt.show()